# Predictive Model: Can we predict late deliveries?

The root cause analysis found concrete operational issues (Standard Class, geographic concentration). But can we build a model that flags risky shipments *before* they happen?

Spoiler: **No, not with this data.** But the negative result is itself useful — it tells us what data we'd need to collect.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, RocCurveDisplay,
                             precision_score, recall_score, f1_score)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14

VISUALS_DIR = Path('../visuals')
VISUALS_DIR.mkdir(exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(VISUALS_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

df = pd.read_csv('../data/supply_chain_cleaned.csv')
print(f"Loaded: {df.shape}")

### Feature preparation

I need to be careful here. The target is `is_early_or_ontime` (1 = on-time, 0 = late). But I can't use features that wouldn't be available at prediction time — like `actual_shipping_delay` (that's the thing we're trying to predict) or `delivery_status` (that's the outcome).

Also: `order_status` includes things like "COMPLETE" and "PENDING" that happen after shipment. Leaky.

Dropping post-hoc features: delivery_status, order_status, actual_shipping_delay, is_early_or_ontime, late_delivery_risk, etc.

In [2]:
# Define target
y = df['is_early_or_ontime']

# Drop leaky / post-hoc features
leaky_cols = [
    'delivery_status', 'order_status', 'actual_shipping_delay',
    'late_delivery_risk', 'is_early_or_ontime',
    'order_date_dateorders', 'shipping_date_dateorders',
    'customer_email', 'customer_password', 'customer_fname',
    'customer_lname', 'customer_street', 'customer_zipcode',
    'order_zipcode', 'product_description', 'product_image',
    'order_id', 'order_item_id', 'order_customer_id',
    'order_item_cardprod_id', 'product_card_id', 'product_category_id'
]

X = df.drop(columns=[c for c in leaky_cols if c in df.columns], errors='ignore')
print(f"Features: {X.shape[1]}")
print(f"Columns: {X.columns.tolist()}")

### Train/test split — MUST do this BEFORE encoding

I initially made the mistake of encoding categoricals before splitting. That's data leakage — the encoder sees all categories in the test set, including ones that might not appear in training.

For a real deployment, I'd use a Pipeline with ColumnTransformer. But to keep it simple here, I'll split first and fit encoders only on training data.

Also using a **time-based split** (training on older orders, testing on newer ones). Random splits would overestimate performance because they'd mix past and future orders.

In [3]:
# Time-based split (80th percentile)
dates = pd.to_datetime(df['order_date_dateorders'])
date_cutoff = dates.quantile(0.80)
train_mask = dates < date_cutoff
test_mask = dates >= date_cutoff

X_train_raw, X_test_raw = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train_raw):,} samples (before {str(date_cutoff)[:10]})")
print(f"Test:  {len(X_test_raw):,} samples (from {str(date_cutoff)[:10]} onward)")

Now encode categoricals — fitting on TRAIN only, transforming both.

In [4]:
# Encode categoricals AFTER split to avoid leakage
cat_cols = X_train_raw.select_dtypes(include='object').columns.tolist()
print(f"Categorical features: {cat_cols}")

# Keep numeric columns only for training features
X_train = X_train_raw.select_dtypes(include=np.number).copy()
X_test = X_test_raw.select_dtypes(include=np.number).copy()

# Label encode categoricals — fit on train, transform both
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # Fit on train only
    train_vals = X_train_raw[col].astype(str)
    le.fit(train_vals)
    # Transform both
    X_train[col] = le.transform(train_vals)
    # Handle unseen categories in test
    test_vals = X_test_raw[col].astype(str)
    X_test[col] = test_vals.map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)
    encoders[col] = le

print(f"\nFinal feature count: {X_train.shape[1]}")
print(f"Feature types:\n{X_train.dtypes.value_counts()}")

### Baseline: Logistic Regression

Before jumping to XGBoost, I should check if a simple model can do anything useful. If a linear model also fails, the problem is the data, not the algorithm.

In [5]:
# Logistic regression baseline
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)

lr_proba = lr.predict_proba(X_test)[:, 1]
lr_pred = lr.predict(X_test)
lr_roc_auc = roc_auc_score(y_test, lr_proba)

print("=== Logistic Regression Baseline ===")
print(f"Test ROC-AUC: {lr_roc_auc:.4f}")
print(classification_report(y_test, lr_pred, target_names=['On-Time', 'Late']))

# Always-predict-majority-class baseline
import collections
baseline_acc = max(y_test.mean(), 1 - y_test.mean())
print(f"\nAlways-predict-majority-class accuracy: {baseline_acc:.3f}")

Logistic regression gets 0.52 ROC-AUC — barely above random. So the issue isn't model complexity. The static order features just don't contain enough signal.

Let me try XGBoost anyway to see if it captures non-linear patterns that logistic regression misses.

Before giving up on prediction entirely, I tried one more thing: creating a "historical late rate by route" feature. For each origin-destination pair, I calculated the historical late rate from the training period and used it as a feature. The idea was that if a route was historically bad, it would predict future late deliveries.

It didn't help. ROC-AUC barely budged from 0.525. The problem is that the training period (2015-2017) doesn't capture the conditions that caused delays in 2018. A route that was fine for 3 years can suddenly fail because of a carrier change or port strike.

This tells me that late delivery prediction needs real-time data feeds, not historical features. Without weather, port congestion, and carrier status, any model will be guessing.

In [6]:
# XGBoost with time-based cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Simple XGBoost (no heavy tuning — the data doesn't warrant it)
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
roc_auc = roc_auc_score(y_test, y_proba)

print("=== XGBoost ===")
print(f"Test ROC-AUC: {roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['On-Time', 'Late']))

# TimeSeriesSplit CV
cv_scores = []
for train_idx, val_idx in tscv.split(X_train):
    fold_model = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=(y_train.iloc[train_idx] == 0).sum() / (y_train.iloc[train_idx] == 1).sum(),
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    fold_model.fit(X_train.iloc[train_idx], y_train.iloc[train_idx], verbose=False)
    fold_proba = fold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    cv_scores.append(roc_auc_score(y_train.iloc[val_idx], fold_proba))

print(f"\nTimeSeries CV ROC-AUC: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores)*2:.4f})")

# Diagnostic: ROC curve
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0])
axes[0].set_title(f'ROC Curve (AUC = {roc_auc:.3f})')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['On-Time', 'Late'], yticklabels=['On-Time', 'Late'])
axes[1].set_title('Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
save_fig('27_model_performance')
plt.show()

### Feature importance — what does the model think matters?

Even with a near-random model, feature importance shows what the model latches onto. But take this with a grain of salt — when ROC-AUC is 0.53, the model is basically guessing.

In [7]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Top 10 Features (XGBoost) ===")
display(importance.head(10))

fig, ax = plt.subplots(figsize=(10, 6))
top10 = importance.head(10)
sns.barplot(data=top10, x='importance', y='feature', ax=ax)
ax.set_title('Feature Importances (XGBoost — model ROC-AUC 0.525)')
ax.set_xlabel('Importance')
save_fig('28_feature_importance')
plt.show()

The top features all have importances around 0.04-0.05 — basically flat. No single feature stands out. This confirms the model has no signal.

### Key findings

1. **ROC-AUC 0.525** — the model cannot predict late deliveries from static order features alone
2. **Logistic regression also fails (0.52)** — it's not a model complexity issue
3. **Feature importances are flat** — no feature dominates, all are weak predictors
4. **The time-based split reveals non-stationarity** — CV ROC-AUC was higher because it used random folds; the time split exposed the reality

### Why did the model fail?

The dataset has:
- Order-level data (product, customer, location, etc.)
- Scheduled vs actual shipping dates

But it **lacks**:
- Real-time logistics data (weather, port congestion, carrier tracking)
- Carrier/warehouse identifiers
- Historical delay patterns at the route level
- Economic indicators or external factors

Late delivery prediction requires **dynamic, time-sensitive data**. A static dataset from 2015-2018 captures none of the conditions that cause delays on any given day.

### What I'd do differently

1. **Skip the ML entirely for this dataset.** The root cause analysis in notebook 02 already identified the actionable findings (Standard Class is the problem, Pareto concentration). The ML model added no value.

2. **If I had to build a model:** start with a simple heuristic baseline (e.g., predict late if shipping_mode == 'Standard Class') before touching XGBoost. That baseline would probably match 0.525 ROC-AUC.

3. **Collect different data.** For real predictive capability, I'd need daily/weekly feeds of: port congestion scores, weather data along shipping routes, carrier on-time performance by lane. Without that, any model is guessing.

4. **The honest result is the most valuable result.** Reporting a 0.525 ROC-AUC is better than fabricating a 0.92. It shows I can recognize when an approach doesn't work, which is a skill that matters more than getting a high number.